# 08 - Reinforcement Learning

## What / Why / How

**What we are trying to do:** Learn how reward-driven training works using Q-learning in a small navigation world.

**Why this matters:** Reinforcement learning is powerful, but robotics makes exploration expensive and safety-critical. This notebook shows the core idea in a safe toy setting.

**How we will do it:** Define states, actions, rewards, and updates, train a Q-table, then evaluate the learned greedy route.

## Goal

Reinforcement learning trains from reward, not demonstrations.

RL can discover strategies, but robotics RL is expensive because robots are slow, fragile, and safety constrained. This is why RL is usually done in simulation first.

You will implement Q-learning in a small gridworld.

In [ ]:
import math
import random
from collections import defaultdict

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see the plot: pip install -r requirements.txt")

## Gridworld Environment

In [ ]:
H, W = 7, 9
start = (0, 0)
goal = (6, 8)
obstacles = {(2, 2), (2, 3), (2, 4), (4, 5), (5, 5), (1, 7)}
actions = [(1,0), (-1,0), (0,1), (0,-1)]

def step_env(state, action_idx):
    r, c = state
    dr, dc = actions[action_idx]
    nr, nc = r + dr, c + dc
    if nr < 0 or nr >= H or nc < 0 or nc >= W or (nr, nc) in obstacles:
        nr, nc = r, c
        reward = -1.0
    else:
        reward = -0.05
    done = (nr, nc) == goal
    if done:
        reward = 5.0
    return (nr, nc), reward, done

print("states:", H * W, "actions:", len(actions))

## Q-Learning

In [ ]:
rng = np.random.default_rng(9)
Q = np.zeros((H, W, len(actions)))
alpha = 0.3
gamma = 0.95
epsilon = 0.25
episode_returns = []

for episode in range(600):
    state = start
    total = 0.0
    for _ in range(100):
        if rng.random() < epsilon:
            action_idx = rng.integers(len(actions))
        else:
            action_idx = int(np.argmax(Q[state[0], state[1]]))
        next_state, reward, done = step_env(state, action_idx)
        total += reward
        best_next = np.max(Q[next_state[0], next_state[1]])
        td_target = reward + gamma * best_next * (not done)
        td_error = td_target - Q[state[0], state[1], action_idx]
        Q[state[0], state[1], action_idx] += alpha * td_error
        state = next_state
        if done:
            break
    episode_returns.append(total)
    epsilon = max(0.03, epsilon * 0.995)

print("last 20 episode return mean:", round(float(np.mean(episode_returns[-20:])), 3))

## Evaluate Learned Policy

In [ ]:
def greedy_route(max_steps=50):
    state = start
    route = [state]
    total = 0.0
    for _ in range(max_steps):
        a = int(np.argmax(Q[state[0], state[1]]))
        state, reward, done = step_env(state, a)
        route.append(state)
        total += reward
        if done:
            break
    return route, total

route, total = greedy_route()
print("route length:", len(route), "return:", round(total, 3), "success:", route[-1] == goal)
print(route)

In [ ]:
if HAS_PLOT:
    canvas = np.zeros((H, W))
    for r, c in obstacles:
        canvas[r, c] = -1
    for r, c in route:
        canvas[r, c] = 0.5
    canvas[goal] = 1
    plt.figure(figsize=(6, 4))
    plt.imshow(canvas, origin="upper", cmap="viridis")
    plt.title("Q-learning route")
    plt.xticks(range(W))
    plt.yticks(range(H))
    plt.grid(color="white", alpha=0.3)
    plt.show()
else:
    plot_unavailable()

## Robotics Reality Check

RL is powerful, but physical robots make it hard:

- Real-world exploration can break hardware.
- Rewards are hard to design.
- Training needs many trials.
- Simulators do not perfectly match reality.

This is why many practical systems use imitation first, then RL for refinement in simulation, then careful real-world adaptation.

## Exercises

1. Make the reward sparse: only reward the goal.
2. Add random action slip.
3. Compare Q-learning with the A* planner from notebook 6.
4. Describe what would make this unsafe on a real robot.